# DuckDB Masterclass — Capítulo 03
## Tipos aninhados: `LIST`, `STRUCT`, `MAP` e JSON nativo

**Edição-piloto para autoaprendizagem e publicação** · [Laboratórios no GitHub](https://github.com/aestevaomoraes/duckdb-masterclass)

**Público:** pessoas com SQL intermediário que desejam modelar dados semiestruturados no DuckDB.

**Como estudar:** leia a hipótese → execute a célula → confira a validação → interprete os resultados. Os registros são **inteiramente sintéticos**; não há credenciais nem caminhos pessoais.

**Pré-requisitos:** Python, Jupyter, DuckDB e Pandas. Execute o notebook de cima para baixo em kernel reiniciado.

## 03.1 — Objetivos e diagnóstico

Ao final você deverá ser capaz de: distinguir os quatro tipos; selecionar campos de `STRUCT`; agregar e expandir `LIST`; consultar chaves de `MAP`; extrair propriedades de JSON; e escolher um tipo adequado a uma necessidade operacional.

**Pergunta diagnóstica:** como representar três medições de latência de um equipamento, sua localização e atributos variáveis, sem transformar tudo em uma única string?

## 03.2 — Modelo mental

| Tipo | Pergunta que responde | Exemplo de operação |
|---|---|---|
| `STRUCT` | Quais atributos *fixos* pertencem ao mesmo registro? | `local.cidade` |
| `LIST` | Quais valores se repetem dentro de uma linha? | `list_sum(latencias)` |
| `MAP` | Quais atributos são pares chave–valor tipados? | `map_extract_value(tags, 'prioridade')` |
| `JSON` | Como ler um documento semiestruturado? | `json_extract_string(payload, '$.modelo')` |

**Importante:** JSON é flexível, mas não substitui automaticamente os tipos aninhados tipados; conversões e campos ausentes precisam de tratamento.

### 🗺️ Anatomia de um equipamento

```text
Equipamento 101
├── local: STRUCT {cidade: 'Campo Grande', uf: 'MS'}
├── latencias: LIST [10, 20, 30]
├── tags: MAP {'prioridade': 'alta', 'ambiente': 'campo'}
└── payload: JSON {"modelo":"OLT-X","portas":8}
```

Cada estrutura atende a um padrão de acesso diferente. A imagem mental é a de um registro com campos tipados, não de uma string única.

## 03.3 — Preparação reproduzível

Não há download de dados ou instalação automática. Caso o import falhe, instale `duckdb` e `pandas` no ambiente do kernel antes de reexecutar.

In [ ]:
import duckdb
from IPython.display import display

print("DuckDB:", duckdb.__version__)
con = duckdb.connect(":memory:")


## 03.4 — Laboratório A: `STRUCT` e `LIST`

**Cenário:** uma equipe analisa latências de dois equipamentos por município.

**Hipótese:** acessar `local.cidade` não exige uma tabela separada; cada `LIST` pode ser agregada dentro do registro.

In [ ]:
con.execute("""
CREATE TABLE equipamentos AS
SELECT 101 AS id,
       {'cidade': 'Campo Grande', 'uf': 'MS'} AS local,
       [10,20,30] AS latencias
UNION ALL
SELECT 102, {'cidade': 'Dourados', 'uf': 'MS'}, [15,25]
""")

res_estruturas = con.execute("""
SELECT id, local.cidade AS cidade,
       list_sum(latencias) AS soma_latencias,
       len(latencias) AS qtd_medicoes
FROM equipamentos ORDER BY id
""").df()
display(res_estruturas)
assert res_estruturas['soma_latencias'].tolist() == [60, 40]
assert res_estruturas['qtd_medicoes'].tolist() == [3, 2]


**Interpretação:** a soma das medições por equipamento é 60 e 40. Essa operação não calcula automaticamente a média por cidade; a granularidade permanece **uma linha por equipamento**.

## 03.5 — Laboratório B: expandindo uma `LIST`

**Hipótese:** `UNNEST` gera uma linha por medição, permitindo análises de granularidade mais fina. O número total esperado é 5.

In [ ]:
medicoes = con.execute("""
SELECT id, unnest(latencias) AS latencia_ms
FROM equipamentos
ORDER BY id, latencia_ms
""").df()
display(medicoes)
assert len(medicoes) == 5
assert medicoes['latencia_ms'].sum() == 100


**Limite:** ao expandir listas, cada elemento vira uma observação. Se houver múltiplas listas por registro, é necessário definir cuidadosamente a regra de alinhamento entre elas.

## 03.6 — Laboratório C: `MAP` para atributos variáveis

**Cenário:** diferentes equipamentos recebem etiquetas operacionais, sem exigir uma coluna para cada etiqueta.

**Hipótese:** é possível consultar um valor pela chave; uma chave ausente não deve ser interpretada como valor confirmado.

In [ ]:
res_mapa = con.execute("""
WITH equipamentos_tags AS (
    SELECT 101 AS id, map(['prioridade', 'ambiente'], ['alta', 'campo']) AS tags
    UNION ALL
    SELECT 102 AS id, map(['prioridade'], ['normal']) AS tags
)
SELECT id, map_extract_value(tags, 'prioridade') AS prioridade,
       map_extract_value(tags, 'ambiente') AS ambiente
FROM equipamentos_tags
ORDER BY id
""").df()
display(res_mapa)
assert res_mapa['prioridade'].tolist() == ['alta', 'normal']
assert res_mapa['ambiente'].isna().sum() == 1


**Interpretação:** a ausência de `ambiente` no equipamento 102 resulta em valor nulo, não em `campo` nem em uma etiqueta inferida.

## 03.7 — Laboratório D: JSON nativo

**Cenário:** recebemos documentos JSON com modelos e quantidade de portas.

**Hipótese:** podemos extrair atributos nomeados e realizar conversões explícitas de tipo.

In [ ]:
res_json = con.execute(r"""
WITH json_equipamentos AS (
    SELECT 101 AS id, '{"modelo":"OLT-X","portas":8}'::JSON AS payload
    UNION ALL
    SELECT 102, '{"modelo":"OLT-Y","portas":16}'::JSON
)
SELECT id,
       json_extract_string(payload, '$.modelo') AS modelo,
       TRY_CAST(json_extract_string(payload, '$.portas') AS INTEGER) AS portas
FROM json_equipamentos
ORDER BY id
""").df()
display(res_json)
assert res_json['modelo'].tolist() == ['OLT-X', 'OLT-Y']
assert res_json['portas'].tolist() == [8, 16]


**Interpretação:** `json_extract_string` recupera o texto; `TRY_CAST` transforma a quantidade em inteiro quando possível. Valores ausentes ou incompatíveis precisam ser investigados, pois podem produzir nulos.

## 03.8 — Experimento controlado e desafio profissional

**Questão de negócio:** uma equipe deseja sinalizar equipamentos com média de latência acima de 19 ms.

**Antes de abrir a solução:** calcule a média das medições por equipamento; documente por que escolheu essa granularidade e como trataria uma lista vazia.

**Critérios de aceite:** uma linha por equipamento, coluna `media_latencia_ms`, validação automática e interpretação em linguagem operacional.

In [ ]:
# Solução de referência — execute somente após tentar
res_desafio = con.execute("""
SELECT id, local.cidade AS cidade,
       ROUND(list_avg(latencias), 2) AS media_latencia_ms,
       CASE WHEN list_avg(latencias) > 19 THEN 'INVESTIGAR' ELSE 'NORMAL' END AS acao
FROM equipamentos ORDER BY id
""").df()
display(res_desafio)
assert res_desafio['media_latencia_ms'].tolist() == [20.0, 20.0]
assert res_desafio['acao'].tolist() == ['INVESTIGAR', 'INVESTIGAR']


**Discussão:** ambas as médias são 20 ms; o filtro foi estritamente `>19`. Esse pequeno conjunto serve para validar lógica, não um SLA real. Para produção, seria preciso especificar janela temporal, amostragem, qualidade de sinal e regra de alerta.

## 03.9 — Autoavaliação (responda antes de ler o gabarito)

1. Quando `STRUCT` é mais adequado que um documento JSON arbitrário?
2. O que muda na granularidade quando usamos `UNNEST`?
3. Qual a diferença entre um `MAP` sem a chave e uma string vazia?
4. Por que converter explicitamente valores extraídos de JSON?
5. Como verificaria se um alerta de latência foi calculado por equipamento e não por amostra?

<details><summary>Gabarito comentado — abra após responder</summary>

1. Quando os campos possuem estrutura conhecida e devem ser consultados com tipos definidos.
2. Uma linha pode gerar várias observações; métricas e contagens mudam de granularidade.
3. Uma chave ausente representa ausência de valor; string vazia é um valor presente.
4. Porque dados extraídos como texto não garantem tipo numérico para cálculo.
5. Comparando contagens antes/depois, agrupamento por ID e médias calculadas manualmente nos registros de teste.

</details>

## 03.10 — Referências, limites e publicação

- [DuckDB — Data Types](https://duckdb.org/docs/stable/sql/data_types/overview)
- [DuckDB — LIST](https://duckdb.org/docs/stable/sql/data_types/list)
- [DuckDB — STRUCT](https://duckdb.org/docs/stable/sql/data_types/struct)
- [DuckDB — MAP](https://duckdb.org/docs/stable/sql/data_types/map)
- [DuckDB — JSON](https://duckdb.org/docs/stable/data/json/overview)
- [DuckDB — UNNEST](https://duckdb.org/docs/stable/sql/query_syntax/unnest)

**Reprodução:** reinicie o kernel, execute todas as células de cima para baixo e salve as saídas esperadas. Não publique identificadores do sistema local ou dados reais.

**Status editorial:** piloto preparado; **execução integral no DuckDB não verificada neste ambiente**. Valide no kernel do projeto antes de publicar.